#  ```insert_subgraph_with_mappings```
This example demonstrates how to use the ``insert_subgraph_with_mappings`` API to simplify graph surgery and model optimization tasks. Specifically, we will focus on converting a ``MatMul`` layer to a ``Conv`` layer. This tutorial will show you how to extract constant variables, such as weights, from an existing graph and reuse them in an optimized subgraph using the API. For clarity, we will concentrate on a single case where the input tensor is 2D with shape ``[w, h]`` and ``h != 1``.
<br><b>This method requires PyTorch to be installed, as it uses PyTorch to generate the replacement ONNX subgraph.</b>

## <b>Setup</b>

In [ ]:
%pip install torch onnx onnx-graphsurgeon numpy onnxruntime torch netron

In [6]:
# Step 1: Setup and Imports
import onnx
import onnx_graphsurgeon as gs
import numpy as np
import logging

import tempfile
import os
import torch
import torch.nn as nn

# Optional: Set logging level for easier debugging
logging.basicConfig(level=logging.INFO)

## <b>Problem</b>
This solution scans the ONNX graph for ``MatMul`` nodes with constant weights and replaces each one with a functionally equivalent ``Conv1x1`` subgraph. For every MatMul found, it extracts and transposes the weights, reshapes the input to ``4D``, applies a ``Conv1x1`` operation, and then reshapes the output back to the original 2D shape. The original MatMul node is then replaced with this new subgraph, ensuring the model’s output remains unchanged.

## <b>Code Flow</b>
Below is a step-by-step example showing how you can use this API to customize your ONNX graphs for your own needs. With just a basic understanding of ONNX GraphSurgeon/ Pytorch, you can easily perform targeted graph surgery and optimize your models, no need to retrain or rebuild from scratch!

### <b>Step 0 : Create/Load the model</b>
Load or create your model which needs to be modified

In [7]:
model = onnx.load("./example_models/complex_matmul_graphsurgeon.onnx") # <- Replace with your ONNX model path
graph = gs.import_onnx(model)

import netron
# Launch Netron to visualize the ONNX model in your browser
netron.start("./example_models/complex_matmul_graphsurgeon.onnx")

INFO:netron.server:Serving './example_models/complex_matmul_graphsurgeon.onnx' at http://localhost:24104


('localhost', 24104)

### <b>Step 1 : Load the API</b>
Load the API from ```common.py```

In [8]:
from osrt_model_tools.onnx_tools.tidl_onnx_model_optimizer.src.common import insert_subgraph_with_mappings

### <b>Step 2(a) : Use PyTorch to Create the Replacement Subgraph(Suggested) </b>
You can leverage PyTorch to define the new subgraph. The following method demonstrates how to use PyTorch to construct the replacement subgraph, highlighting important considerations and best practices for this approach.

#### <b> Step 2.1(a) : Define a custom Transformation Function </b>

In [10]:
def matmul_to_conv2d_pytorch(graph: gs.Graph):
    """
    Replace all MatMul nodes with constant weights with an equivalent Conv2d subgraph (NCHW),
    using PyTorch to generate the new subgraph and injecting the extracted weights.
    Handles 2D input [h, w] with h != 1 by transposing, reshaping, Conv2d, and reshaping back.
    (Ni*No matmul -> No*Ni*1*1 Conv2d, where No is the output features and Ni is the input features.)
    """
    nodes = graph.nodes
    idx = 0
    for node in nodes:
        if node.op == "MatMul" and isinstance(node.inputs[1], gs.Constant):
            input_name = node.inputs[0].name
            output_name = node.outputs[0].name
            input_shape = node.inputs[0].shape  # [h, w] or [h, in_features]
            output_shape = node.outputs[0].shape  # [h, out_features]
            suffix = f"_matmul2conv2d_{idx}"
            
            # used for generating the Conv2d module
            h, w = input_shape[-2], input_shape[-1]
            in_features = w                     # in_features = w
            out_features = output_shape[-1]

            # --- 1. Extract weights from the original MatMul node ---
            matmul_weight = np.array(node.inputs[1].values, dtype=np.float32)  # [in_features, out_features]
            conv_weight = matmul_weight.T[:, :, None, None]  # [out_features, in_features, 1, 1] [out_channel, in_channel, kernel_h, kernel_w]

            # --- 2. Dynamically create a Conv2d PyTorch module and load the weights ---
            class LinearToConv2d(nn.Module):
                def __init__(self, in_features, out_features, weight):
                    super().__init__()
                    self.conv = nn.Conv2d(
                        in_channels=in_features,
                        out_channels=out_features,
                        kernel_size=(1, 1),
                        bias=False
                    )
                    with torch.no_grad():
                        self.conv.weight.copy_(torch.from_numpy(weight))
                def forward(self, x):
                    x = x.permute(1,0)  # [h,w] -> [w,h]
                    x = x.reshape(1, w, h, 1)  # [1, w, h, 1]
                    y = self.conv(x)  # [1, out_features, h, 1]
                    y = y.reshape(out_features, h)  # [out_features, h]
                    y = y.permute(1, 0)  # [h, out_features]
                    return y

            dummy_input = torch.randn(*input_shape)
            model = LinearToConv2d(
                in_features=in_features,
                out_features=out_features,
                weight=conv_weight
            )

            # --- 3. Export the PyTorch model to ONNX ---
            with tempfile.NamedTemporaryFile(suffix=".onnx", delete=False) as tmpfile:
                torch.onnx.export(
                    model,
                    dummy_input,
                    tmpfile.name,
                    input_names=["input"],
                    output_names=["output"],
                    dynamic_axes=None,
                    opset_version=graph.opset
                )
                conv_model = onnx.load(tmpfile.name)
                conv_model = onnx.shape_inference.infer_shapes(conv_model)
            os.remove(tmpfile.name)

            # --- 4. Import the ONNX subgraph into GraphSurgeon ---
            conv_gs = gs.import_onnx(conv_model)

            input_mapping = {conv_gs.inputs[0].name: input_name}
            output_mapping = {conv_gs.outputs[0].name: output_name}


            # --- 5. Use the API to insert the new subgraph in place of the MatMul node ---
            success = insert_subgraph_with_mappings(
                graph,
                input_mapping,
                output_mapping,
                conv_gs,
                suffix
            )
            if success:
                print(f"MatMul→Conv2d replacement succeeded for node {node.name}.")
            else:
                print(f"MatMul→Conv2d replacement failed for node {node.name}.")
                
            idx += 1 # important to increment idx to avoid name collisions


#### <b>Step 2.2(a) : Call the function</b>
Pass the original graph to the function and save the transformed graph, if required

In [11]:
matmul_to_conv2d_pytorch(graph)
onnx.save(onnx.shape_inference.infer_shapes(gs.export_onnx(graph)), "./example_models/optim_complex_matmul_graphsurgeon.onnx")
netron.start("./example_models/optim_complex_matmul_graphsurgeon.onnx")

INFO:netron.server:Serving './example_models/optim_complex_matmul_graphsurgeon.onnx' at http://localhost:15904


MatMul→Conv2d replacement succeeded for node .
MatMul→Conv2d replacement succeeded for node .


('localhost', 15904)

### <b>Step 2(b) : Use GraphSurgeon to Create the Replacement Subgraph(Advanced Users) </b>
You can leverage GraphSurgeon to define the new subgraph. The following method demonstrates how to use GraphSurgeon to construct the replacement subgraph, highlighting important considerations and best practices for this approach. Use this approach if you want to 
insert specific nodes with specific attributes.

#### <b>Step 2.1(b) : Define a class</b> (Optional)
Defining a class for your replacement pattern is a good practice. It helps keep your code modular and organized, making it easier to visualize and manage all the important details needed for optimization. While this step is optional, it’s highly recommended for clarity and maintainability, especially as your graph surgery tasks become more complex.

In [6]:
class MatMulToConvPattern:
    def __init__(self):
        # Used for identifying the pattern instance.
        self.idx = -1
        
        self.input = None
        self.input_shape = None
        
        self.weight = -1 # <-- This will be set to the weight tensor of the MatMul.
        
        self.output = None
        self.output_shape = None
        
        self.replace_ready = False
        
        # Used for the API
        self.new_subgraph = None
        self.input_mapping = None
        self.output_mapping = None

    def optimize(self, graph: gs.Graph):
        """
        Build a new subgraph (Transpose/Reshape + Conv + Reshape/Transpose) to replace MatMul.
        Step 1: Extract and prepare weights for Conv.
        """
        # Give unique names to add the suffix for each pattern instance.
        suffix = f"_{self.idx}"
        
        # Extract the MatMul input, weight, and output names and shapes.
        input_name = self.input
        weights = self.weight
        output_name = self.output
        # In graph surgeon, it's necessary to provide the shapes for the input and output tensors explicitly.
        input_shape = self.input_shape
        output_shape = self.output_shape

        weights = weights.T  # [in, out] -> [out, in]
        
        # use the extracted weights to create a Constant for Conv. !IMP
        kernel = np.reshape(weights, (weights.shape[0], weights.shape[1], 1, 1))
        conv_weight_const = gs.Constant(f"{self.output}_conv", values=kernel)
        
        # Reshape [h, w] → [h, w, 1, 1] (treat each row as a batch).
        in_var = gs.Variable(input_name, dtype=np.float32, shape=input_shape)
        reshape1_out = gs.Variable(f"{input_name}_reshape1{suffix}", dtype=np.float32)
        reshape1_shape = gs.Constant(
            f"{input_name}_reshape1_shape{suffix}",
            values=np.array([input_shape[0], input_shape[1], 1, 1], dtype=np.int64)
        )
        reshape1_node = gs.Node(
            op="Reshape",
            name=f"{input_name}_reshape1_node",
            inputs=[in_var, reshape1_shape],
            outputs=[reshape1_out]
        )
        
        # Create the Conv node with 1x1 kernel and stride, using the prepared weights
        conv_out = gs.Variable(f"{output_name}_conv{suffix}", dtype=np.float32)
        conv_node = gs.Node(
            op="Conv",
            name=f"{output_name}_conv_node{suffix}",
            attrs={"kernel_shape": [1, 1], "strides": [1, 1]},
            inputs=[reshape1_out, conv_weight_const],
            outputs=[conv_out]
        )
        
        
        # s4 : The Conv output will be [h, out_features, 1, 1].
        # we want [h, out_features] (original MatMul output).
        reshape2_out = gs.Variable(output_name, dtype=np.float32, shape=output_shape)
        reshape2_shape = gs.Constant(
            f"{output_name}_reshape2_shape{suffix}",
            values=np.array([input_shape[0], weights.shape[0]], dtype=np.int64)  # weights.shape[0] is out_features after transpose!
        )
        reshape2_node = gs.Node(
            op="Reshape",
            name=f"{output_name}_reshape2_node{suffix}",
            inputs=[conv_out, reshape2_shape],
            outputs=[reshape2_out]
        )
        
        # Create the new subgraph with the nodes.
        self.new_subgraph = gs.Graph(
            nodes=[reshape1_node, conv_node, reshape2_node],
            inputs=[in_var],
            outputs=[reshape2_out]
        )
        
        # map the new subgraph inputs and outputs to the original MatMul input and output.
        self.input_mapping = {self.input: self.input}
        self.output_mapping = {self.output: self.output}
        
        print(f"Input mapping: {self.input_mapping}")
        print(f"Output mapping: {self.output_mapping}")
        
        # Mark the pattern as ready for replacement.
        self.replace_ready = True

- ***Attributes***
    - ``idx``: Index to uniquely identify each pattern instance.
    - ``input``: Name of the input tensor to the MatMul node.
    - ``input_shape``: Shape of the input tensor.
    - ``weight``: The constant weight tensor extracted from the MatMul node.
    - `` output``: Name of the output tensor from the MatMul node.
    - ``output_shape``: Shape of the output tensor.
    - ``replace_ready``: Boolean flag indicating if the replacement subgraph is ready.
    - ``new_subgraph``: The ONNX GraphSurgeon subgraph that will replace the original MatMul node.
    - ``input_mapping``: Dictionary mapping the new subgraph’s input tensor names to the original graph’s input tensor names.
    - ``output_mapping``: Dictionary mapping the new subgraph’s output tensor names to the original graph’s output tensor names.
- ***``optimize`` Method***
    - **Purpose**:
    <br>
        Builds a new subgraph that replaces a MatMul node with a functionally equivalent Conv1x1 operation.
    - **Steps**:
        1. Extracts and prepares weights:
        <br>
            Transposes and reshapes the MatMul weights to match Conv’s expected format.
        2. Reshapes the input:
        <br>
            Converts the 2D input tensor ``[h, w]`` to a 4D tensor ``[h, w, 1, 1]`` for Conv compatibility.
        3. Creates the Conv node:
        <br>
            Adds a Conv node with a 1x1 kernel and the prepared weights.
        4. Restores the output shape:
        <br>
            Reshapes the Conv output back to the original 2D output shape ``[h, out_features]``.
        5. Packages the subgraph:
        <br>
            Assembles the new nodes into a subgraph and sets up input/output mappings for seamless replacement.

#### <b>Step 2.2(b) : Define a Transformation Function</b>
Define a function which stores the various attributes of ```MatMul``` Node in the ```MatMulToConvPattern``` Class

In [ ]:
def tidl_matmul_to_conv1x1(graph: gs.Graph):
    """
    Replace all MatMul nodes with an equivalent Conv1x1 subgraph using the API.
    """
    # [h, w] (2D, h != 1) case
    nodes = graph.nodes
    num_patterns = 0
    patterns = []

    for idx, node in enumerate(nodes):
        if node.op == "MatMul" and isinstance(node.inputs[1], gs.Constant):
            pattern = MatMulToConvPattern()
            pattern.idx = num_patterns
            pattern.input = node.inputs[0].name
            pattern.input_shape = node.inputs[0].shape
            pattern.output = node.outputs[0].name
            pattern.output_shape = node.outputs[0].shape
            num_patterns += 1
            
            pattern.weight = np.array(node.inputs[1].values, dtype=np.float32)

            pattern.optimize(graph)
            patterns.append(pattern)

            if pattern.replace_ready:
                success = insert_subgraph_with_mappings(
                    graph,
                    pattern.input_mapping,
                    pattern.output_mapping,
                    pattern.new_subgraph
                )
                if success:
                    print("MatMul→Conv1x1 replacement succeeded.")
                else:
                    print("Replacement failed due to dependencies or shape mismatch.")

    onnx.save(gs.export_onnx(graph), "./example_models/complex_matmul_conv1x1.onnx")  # <- Replace with your desired output path
    netron.start("./example_models/complex_matmul_conv1x1.onnx")

# call the function
tidl_matmul_to_conv1x1(graph)